<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/runners/runner_flexibility_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Runner flexibility: Lightning steps with Lightning, custom steps without

The same tiny workflow three ways: `Trainer.predict` for Lightning-supported steps,
`AnalysisRunner` for custom analysis steps, and a raw PyTorch loop for everything else.
Mix and match within one workflow — the framework does not care which driver runs which step.

Runs on CPU with the ungated `rte_demo.gpt2.core` config; no GPU, token, or gated weights.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
MAX_EXAMPLES = 8
SEED = 0
REVISION = (
    None  # component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)

In [3]:
# @title Imports { display-mode: "form" }
import interpretune as it
import interpretune.analysis  # noqa: F401  (registers the it.<op> wrappers)
import torch
from lightning.pytorch import Trainer

from it_examples.seeds import ensure_local_seeds

torch.manual_seed(SEED)
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache

{'speediedan/prompt-configs': 'local98039ddc1f387a9eb0de0838f33bc6703c4'}

## A. Lightning-supported steps run with Lightning

`trainer.fit` drives the module's training step through Lightning. (Predict has a separate,
known gap: direct forward never maps the bridge `input` batch key to model inputs — see the
tracking issue linked from #55. Analysis-routed predict, which goes through the backend that
does map keys, is covered by its own tests.)

In [4]:
# @title A: Trainer.fit { display-mode: "form" }
def make_session(adapter_ctx):
    base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_demo.gpt2.core", revision=REVISION)
    cfg = it.ITSessionConfig(
        adapter_ctx=adapter_ctx,
        datamodule_cfg=base_itdm_cfg,
        module_cfg=base_it_cfg,
        datamodule_cls=dm_cls,
        module_cls=m_cls,
    )
    return it.ITSession(cfg)


def make_trainer(**kw):
    return Trainer(
        accelerator="cpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        **kw,
    )


sess_l = make_session((it.Adapter.lightning,))
sess_l.datamodule.prepare_data()
sess_l.datamodule.setup(stage="fit")
make_trainer(max_steps=1, limit_train_batches=1).fit(sess_l.module, datamodule=sess_l.datamodule)
print("fit ran: 1 step")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NoneType.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NoneType.forward`, you can safely ignore this message.


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NoneType.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NoneType.forward`, you can safely ignore this message.


...

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

~/.venvs/.../site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
~/.venvs/.../site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


~/.venvs/.../site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
~/.venvs/.../site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 164 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


`Trainer.fit` stopped: `max_steps=1` reached.


fit ran: 1 step


## B. Custom analysis steps run with the AnalysisRunner

Steps Lightning does not support — here an analysis op — run through `AnalysisRunner`, which
generates the analysis step from the config and lands results in an `AnalysisStore`.

In [5]:
# @title B: AnalysisRunner { display-mode: "form" }
from interpretune import AnalysisCfg
from interpretune.config.runner import AnalysisRunnerCfg
from interpretune.runners.analysis import AnalysisRunner

sess_c = make_session((it.Adapter.core,))
sess_c.datamodule.prepare_data()
sess_c.datamodule.setup()
it.it_init(sess_c.module, sess_c.datamodule)
runner = AnalysisRunner(
    run_cfg=AnalysisRunnerCfg(
        it_session=sess_c,
        analysis_cfgs=AnalysisCfg(target_op=it.labels_to_ids, ignore_manual=True),
        max_epochs=1,
    )
)
store_b = runner.run_analysis()
print("store:", type(store_b).__name__)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NoneType.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NoneType.forward`, you can safely ignore this message.


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NoneType.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NoneType.forward`, you can safely ignore this message.


...

Saving the dataset (0/1 shards):   0%|          | 0/139 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: Running analysis end hooks: InterpretunableModule


[INFO] interpretune.utils.logging: Running stage end hooks on IT module: InterpretunableModule


[INFO] interpretune.utils.logging: Running stage end hooks on IT datamodule: InterpretunableDataModule


store: AnalysisStore


## C. Anything else runs in a raw loop

Custom steps that are neither Lightning-supported nor analysis ops — SAE training steps are the
canonical example — run in a plain loop over the same batches. Here a manual forward pass stands
in for one.

In [6]:
# @title C: raw loop { display-mode: "form" }
from interpretune.analysis.ops.base import get_batch_input

seen = 0
sess_c.module.eval()
with torch.no_grad():
    for batch in sess_c.datamodule.test_dataloader():
        if seen >= MAX_EXAMPLES:
            break
        # The task module's own forward computes its classification loss, so a raw loop goes
        # one level down: the bare model on the resolved input ids. Same batches, no machinery.
        ids = get_batch_input(batch)
        take = min(ids.size(0), MAX_EXAMPLES - seen)
        fwd_kwargs = {"input_ids": ids[:take]}
        if "attention_mask" in batch:
            fwd_kwargs["attention_mask"] = batch["attention_mask"][:take]
        _ = sess_c.module.model(**fwd_kwargs)
        seen += take
print("raw-loop examples:", seen)

raw-loop examples: 8


## Takeaway

Three drivers, one workflow: Lightning owns the steps it supports, `AnalysisRunner` owns custom
analysis steps, and a raw loop owns the rest. The batch source and the session are shared; only
the driver changes.